In [17]:
import os
import pandas as pd
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [20]:
# --------------------------------------------------
# 0. 환경 변수 로드
# --------------------------------------------------
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")




In [21]:
# --------------------------------------------------
# 1. 경로 설정
# --------------------------------------------------
RAG_DATA_PATH = "./rag_documents.csv"
PAIR_DATA_PATH = "./response_pairs.csv"

RAG_SAVE_PATH = "./vectorstore/rag"
EXAMPLE_SAVE_PATH = "./vectorstore/example"

In [22]:
# --------------------------------------------------
# 2. 파일 존재 여부 확인
# --------------------------------------------------
def check_file_exists(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"파일을 찾을 수 없습니다: {path}\n"
            f"현재 작업 경로: {os.getcwd()}\n"
            f"현재 폴더 파일 목록: {os.listdir()}"
        )

In [23]:
#--------------------------------------------------
# 3. CSV 로드
# --------------------------------------------------
def load_data():
    check_file_exists(RAG_DATA_PATH)
    check_file_exists(PAIR_DATA_PATH)

    rag_df = pd.read_csv(RAG_DATA_PATH)
    pair_df = pd.read_csv(PAIR_DATA_PATH)

    print(f"rag_documents.csv shape: {rag_df.shape}")
    print(f"response_pairs.csv shape: {pair_df.shape}")

    return rag_df, pair_df

In [24]:
# --------------------------------------------------
# 4. rag_documents.csv -> Document 변환
#    문맥 검색용 문서
# --------------------------------------------------
def build_rag_documents(rag_df):
    rag_docs = []

    for _, row in rag_df.iterrows():
        text = f"""
관계: {row['relation']}
상황: {row['situation']}
화자 감정: {row['speaker_emotion']}
청자 행동: {row['listener_behavior']}

전체 대화:
{row['full_dialogue']}
""".strip()

        doc = Document(
            page_content=text,
            metadata={
                "dialogue_id": row["dialogue_id"],
                "source": "rag_documents"
            }
        )
        rag_docs.append(doc)

    return rag_docs

In [25]:
# --------------------------------------------------
# 5. response_pairs.csv -> Document 변환
#    유사 응답 예시 검색용 문서
# --------------------------------------------------
def build_example_documents(pair_df):
    example_docs = []

    for _, row in pair_df.iterrows():
        text = f"""
관계: {row['relation']}
상황: {row['situation']}
화자 감정: {row['speaker_emotion']}

응답 직전 문맥:
{row['context_before_response']}

청자 응답:
{row['listener_response']}
""".strip()

        doc = Document(
            page_content=text,
            metadata={
                "dialogue_id": row["dialogue_id"],
                "listener_empathy": row.get("listener_empathy", ""),
                "terminate": row.get("terminate", ""),
                "source": "response_pairs"
            }
        )
        example_docs.append(doc)

    return example_docs

In [26]:
# --------------------------------------------------
# 6. Document Chunking
# --------------------------------------------------
def split_documents(docs, chunk_size=500, chunk_overlap=100):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    split_docs = text_splitter.split_documents(docs)
    return split_docs

In [27]:
# --------------------------------------------------
# 7. FAISS Vector Store 생성 및 저장
# --------------------------------------------------
def build_and_save_vectorstore(docs, save_path, embeddings):
    vectorstore = FAISS.from_documents(docs, embeddings)
    vectorstore.save_local(save_path)
    print(f"저장 완료: {save_path}")

In [30]:
def main():
    print("현재 작업 경로:", os.getcwd())
    print("현재 폴더 파일 목록:", os.listdir())

    # ✅ 여기 안에 있어야 함
    from langchain_openai import OpenAIEmbeddings

    # ✅ API KEY 포함 (한 번만!)
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",
        api_key="sk-여기에_너의_API_KEY"
    )

    # 1) 데이터 로드
    rag_df, pair_df = load_data()

    # 2) Document 생성
    rag_docs = build_rag_documents(rag_df)
    example_docs = build_example_documents(pair_df)

    print(f"원본 RAG 문서 수: {len(rag_docs)}")
    print(f"원본 Example 문서 수: {len(example_docs)}")

    # 3) Chunk 분할
    split_rag_docs = split_documents(rag_docs, chunk_size=500, chunk_overlap=100)
    split_example_docs = split_documents(example_docs, chunk_size=500, chunk_overlap=100)

    print(f"분할된 RAG chunk 수: {len(split_rag_docs)}")
    print(f"분할된 Example chunk 수: {len(split_example_docs)}")

    # ❌ 이 줄 삭제해야 함 (중복 생성)
    # embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    # 4) 저장 폴더 생성
    os.makedirs("./vectorstore", exist_ok=True)

    # 5) FAISS vector store 생성 및 저장
    build_and_save_vectorstore(split_rag_docs, RAG_SAVE_PATH, embeddings)
    build_and_save_vectorstore(split_example_docs, EXAMPLE_SAVE_PATH, embeddings)

    print("모든 FAISS vector store 생성이 완료되었습니다.")